In [1]:
import os

In [2]:
%pwd

'e:\\kindey_disease_classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\kindey_disease_classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from kidney_disease_classification.constants import *
from kidney_disease_classification.utils.common import read_yaml,create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        CONFIG_FILE_PATH,
        PARAMS_FILE_PATH
    ):
        self.config=read_yaml(CONFIG_FILE_PATH)
        self.params=read_yaml(PARAMS_FILE_PATH)
        
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self)->DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config
    
        
        

In [11]:
import os
import zipfile
import gdown
from kidney_disease_classification import logger
from kidney_disease_classification.utils.common import get_size

In [17]:
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    def download_file(self)->str:
        try:
            dataset_url=self.config.source_URL
            zip_download_dir=self.config.local_data_file
            os.makedirs("artifacts/data_ingestion",exist_ok=True)
            logger.info(f"downloading data from {dataset_url} into file {zip_download_dir}")
            
            file_id=dataset_url.split("/")[-2]
            prefix='https://drive.google.com/uc?/export/=download&id='
            gdown.download(prefix+file_id,zip_download_dir)
            logger.info(f"downloaded data from {dataset_url} into file {zip_download_dir}")
        except Exception as e:
            raise e
        
    def extract_zip_file(self):
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [18]:
from pathlib import Path

config = ConfigurationManager(
    CONFIG_FILE_PATH=Path("config/config.yaml"),
    PARAMS_FILE_PATH=Path("params.yaml")
)

data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-03-06 20:45:31,196: INFO: common: Yaml File: config\config.yaml load successfully]
[2026-03-06 20:45:31,199: INFO: common: Yaml File: params.yaml load successfully]
[2026-03-06 20:45:31,201: INFO: common: Created directory at: artifacts]
[2026-03-06 20:45:31,202: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-03-06 20:45:31,202: INFO: 2243647136: downloading data from https://drive.google.com/file/d/1g9Box8IlTBnDiIYKWYp3kWXISOFnfyjd/view?usp=drive_link into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export/=download&id=1g9Box8IlTBnDiIYKWYp3kWXISOFnfyjd
From (redirected): https://drive.google.com/uc?%2Fexport%2F=download&id=1g9Box8IlTBnDiIYKWYp3kWXISOFnfyjd&confirm=t&uuid=a0f3d5f4-7e01-4ed2-a2fa-6580cf85959d
To: e:\kindey_disease_classification\artifacts\data_ingestion\data.zip
100%|██████████| 940M/940M [05:10<00:00, 3.03MB/s] 

[2026-03-06 20:50:45,333: INFO: 2243647136: downloaded data from https://drive.google.com/file/d/1g9Box8IlTBnDiIYKWYp3kWXISOFnfyjd/view?usp=drive_link into file artifacts/data_ingestion/data.zip]
